In [5]:
df=pd.read_csv("retail_data.csv",header=None,low_memory=False)

In [17]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet
0,545797,84849D,HOT BATHS SOAP HOLDER,2,2011-03-07 12:17:00,1.69,16225,United Kingdom,2009-2010
1,545797,84978,HANGING HEART JAR T-LIGHT HOLDER,6,2011-03-07 12:17:00,1.25,16225,United Kingdom,2009-2010
2,545797,85099B,JUMBO BAG RED RETROSPOT,3,2011-03-07 12:17:00,1.95,16225,United Kingdom,2009-2010
3,545797,85099F,JUMBO BAG STRAWBERRY,3,2011-03-07 12:17:00,1.95,16225,United Kingdom,2009-2010
4,545797,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-03-07 12:17:00,2.55,16225,United Kingdom,2009-2010


In [8]:
df.columns=['Invoice','StockCode','Description','Quantity','InvoiceDate','Price','Customer_ID','Country','SourceSheet']

In [18]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

print(df.dtypes)

Invoice                object
StockCode              object
Description            object
Quantity               object
InvoiceDate    datetime64[ns]
Price                  object
Customer_ID            object
Country                object
SourceSheet            object
dtype: object


In [22]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet
0,545797,84849D,HOT BATHS SOAP HOLDER,2,2011-03-07 12:17:00,1.69,16225,United Kingdom,2009-2010
1,545797,84978,HANGING HEART JAR T-LIGHT HOLDER,6,2011-03-07 12:17:00,1.25,16225,United Kingdom,2009-2010
2,545797,85099B,JUMBO BAG RED RETROSPOT,3,2011-03-07 12:17:00,1.95,16225,United Kingdom,2009-2010
3,545797,85099F,JUMBO BAG STRAWBERRY,3,2011-03-07 12:17:00,1.95,16225,United Kingdom,2009-2010
4,545797,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-03-07 12:17:00,2.55,16225,United Kingdom,2009-2010


In [21]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

print(df[['Quantity', 'Price', 'InvoiceDate']].dtypes)

Quantity                int64
Price                 float64
InvoiceDate    datetime64[ns]
dtype: object


In [11]:
df = df.drop(index=0).reset_index(drop=True)

In [23]:
#filter and calulcate recenyc,frequency,monetry
rfm_data = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency=(
        'InvoiceDate',
        lambda x: (snapshot_date - x.max()).days
    ),
    Frequency=('Invoice', 'nunique'),
    Monetary=(
        'Quantity',
        lambda x: (
            x * rfm_data.loc[x.index, 'Price']
        ).sum()
    )
).reset_index()

In [24]:
rfm['R_score'] = pd.qcut(
    rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]
).astype(int)

rfm['F_score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5, labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5, labels=[1, 2, 3, 4, 5]
).astype(int)


def segment_customer(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'
    elif row['R_score'] >= 3 and row['F_score'] >= 3:
        return 'Loyal'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] >= 4 and row['F_score'] <= 2:
        return 'New'
    elif row['R_score'] == 3 and row['F_score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Other'


rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm['is_churned'] = rfm['Recency'] > 90

In [28]:
#merge results and export
df = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()
df['Revenue']=df['Quantity']*df['Price']
df['InvoiceMonth']=df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df_final=df.merge(rfm[['Customer_ID','Recency','Frequency','Monetary','R_score','F_score','M_score','Segment','is_churned']],on='Customer_ID',how='left'
)
df_final.to_excel('ecommerce.xlsx',index=False)